# Network Intrusion Detection ML Challenge — End-to-End Solution
**Organized by GDG on Campus**  
**Author:** Senior Machine Learning Engineer & Cybersecurity Data Scientist  
**Primary Metric:** Multiclass Macro F1 Score

---
## Notebook Structure
1. **Environment Setup & Data Loading**
2. **Comprehensive Exploratory Data Analysis & Data Audit**
3. **Leakage-Free Preprocessing & Domain Feature Engineering**
4. **Stratified 5-Fold Cross-Validation Framework**
5. **Model Training & Comparison (LightGBM, CatBoost, XGBoost, ExtraTrees, Random Forest)**
6. **Class Imbalance & Probability Threshold Optimization**
7. **Multi-Model Soft-Voting Ensemble**
8. **Final Submission Generation & Strict Verification**


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(".."))
from src.preprocessing import NetworkPreprocessor, TARGET_MAP, INV_TARGET_MAP
from src.feature_engineering import NetworkFeatureEngineer
from src.evaluate import evaluate_predictions
from src.threshold_optimizer import MulticlassThresholdOptimizer

DATA_DIR = r"C:\Users\yashu\Downloads\network-intrusion-detection2-ml-challenge"
print("Environment initialized successfully.")


In [ ]:
# Load train, test, and sample submission
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print(f"Train Shape: {train_df.shape}")
print(f"Test Shape: {test_df.shape}")
print(f"Sample Submission Shape: {sample_sub.shape}")


In [ ]:
# Analyze Target Class Distribution
target_counts = train_df['Target'].value_counts()
target_pct = train_df['Target'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({'Count': target_counts, 'Percentage (%)': target_pct.round(2)})
print("Target Class Distribution:")
display(dist_df)

plt.figure(figsize=(8, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, palette='viridis')
plt.title("Target Class Distribution (Extreme Class Imbalance)")
plt.ylabel("Number of Records")
plt.yscale("log")
plt.show()


In [ ]:
# Behavioral Differences Across Classes
key_features = [
    'flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'flow_pkts_per_sec',
    'payload_bytes_per_second', 'flow_RST_flag_count', 'flow_ACK_flag_count',
    'fwd_init_window_size', 'bwd_init_window_size'
]
display(train_df.groupby('Target')[key_features].mean().round(2).T)


In [ ]:
# Instantiate leakage-free preprocessing pipeline
prep = NetworkPreprocessor(include_port_freq=True)
fe = NetworkFeatureEngineer()

sample_X = prep.fit_transform(train_df.head(1000))
sample_fe = fe.fit_transform(sample_X)

print(f"Transformed Feature Matrix Shape: {sample_fe.shape}")
print(f"Missing Values: {sample_fe.isnull().sum().sum()}")
print(f"Infinite Values: {np.isinf(sample_fe).sum().sum()}")


In [ ]:
# Load Experiment Results Log
exp_results = pd.read_csv(r"..\reports\experiment_results.csv")
print("Full Experiment Log Summary:")
display(exp_results[['experiment_name', 'model', 'training_time_sec', 'oof_macro_f1_argmax', 'oof_macro_f1_tuned', 'reconnaissance_f1', 'benign_f1', 'spoofing_f1']])


In [ ]:
# Visual Model Comparison
plt.figure(figsize=(10, 5))
sns.barplot(data=exp_results.drop_duplicates('experiment_name'), x='experiment_name', y='oof_macro_f1_tuned', palette='mako')
plt.title("Out-of-Fold Macro F1 Score Comparison Across Models & Ensembles")
plt.ylabel("OOF Macro F1")
plt.ylim(0.97, 0.985)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Winning Ensemble Confusion Matrix
oof_probs = np.load(r"..\models\best_ensemble_oof_probs.npy")
y_true = np.array([TARGET_MAP[val] for val in train_df['Target']])

# Calibrate with optimal weights
opt_weights = [1.0527777777777776, 1.0, 1.0277777777777777, 0.9833333333333334]
scaled_probs = oof_probs * np.array(opt_weights)
oof_preds = np.argmax(scaled_probs, axis=1)

metrics = evaluate_predictions(y_true, oof_preds, verbose=True)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, oof_preds)
class_names = ['Benign', 'Availability', 'Reconnaissance', 'Spoofing']

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title("Winning Ensemble: Out-of-Fold Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.tight_layout()
plt.show()


In [ ]:
# Final Submission Verification
sub_df = pd.read_csv(r"..\submission\submission.csv")
print(f"Submission Shape: {sub_df.shape}")
print(f"Columns: {list(sub_df.columns)}")
print(f"Null count: {sub_df.isnull().sum().to_dict()}")
print(f"ID exact match: {(sub_df['id'] == sample_sub['id']).all()}")

print("\nPredicted Test Class Distribution:")
display(pd.DataFrame({
    'Count': sub_df['Target'].value_counts(),
    'Percentage (%)': (sub_df['Target'].value_counts(normalize=True) * 100).round(2)
}))
